In [12]:
import pandas as pd
import altair as alt

df = pd.read_csv('videogamesales.csv')
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df['Year'] = df['Year'].astype(int)
df = df[(df['Year'] >= 1980) & (df['Year'] <= 2016)]

yearly_sales = (
    df.groupby('Year')['Global_Sales']
    .sum()
    .reset_index()
    .rename(columns={'Global_Sales': 'Total_Global_Sales'}))

brush = alt.selection_interval(encodings=['x'])

hover = alt.selection_point(on='mouseover', nearest=True, fields=['Year'])

line = (
    alt.Chart(yearly_sales)
    .mark_line(point=True, color='#4C78A8', strokeWidth=2.5)
    .encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='Total Global Sales (millions)', scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter(brush)
    .properties(width=700, height=300, title='Global Video Game Sales Over the Years'))

# Vertical rule that follows your mouse
rule = (
    alt.Chart(yearly_sales)
    .mark_rule(color='gray', strokeDash=[4, 4])
    .encode(
        x='Year:O',
        opacity=alt.condition(hover, alt.value(0.8), alt.value(0)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter(brush)
    .add_params(hover))

overview = (
    alt.Chart(yearly_sales)
    .mark_area(color='#4C78A8', opacity=0.3)
    .encode(
        x=alt.X('Year:O', title='', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='', axis=alt.Axis(labels=False))
    )
    .add_params(brush)
    .properties(width=700, height=80, title='Drag to select a year range'))

alt.vconcat(line + rule, overview).configure_view(strokeWidth=0)

alt.VConcatChart(...)

In [11]:
import plotly.express as px

genre_region = (
    df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
    .sum()
    .reset_index())

genre_region['Total'] = genre_region[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum(axis=1)
genre_region['NA_Pct'] = (genre_region['NA_Sales'] / genre_region['Total'] * 100).round(1)
genre_region['EU_Pct'] = (genre_region['EU_Sales'] / genre_region['Total'] * 100).round(1)
genre_region['JP_Pct'] = (genre_region['JP_Sales'] / genre_region['Total'] * 100).round(1)
genre_region['Other_Pct'] = (genre_region['Other_Sales'] / genre_region['Total'] * 100).round(1)

genre_long = genre_region.melt(
    id_vars=['Genre', 'Total', 'NA_Pct', 'EU_Pct', 'JP_Pct', 'Other_Pct'],
    value_vars=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales'],
    var_name='Region',
    value_name='Sales')

region_labels = {
    'NA_Sales': 'North America',
    'EU_Sales': 'Europe',
    'JP_Sales': 'Japan',
    'Other_Sales': 'Other'}

genre_long['Region'] = genre_long['Region'].map(region_labels)

# Map correct pct per row
pct_map = {
    'North America': 'NA_Pct',
    'Europe': 'EU_Pct',
    'Japan': 'JP_Pct',
    'Other': 'Other_Pct'}

genre_long['Pct'] = genre_long.apply(lambda row: row[pct_map[row['Region']]], axis=1)

sort_options = {
    'Total Sales': genre_region.sort_values('Total', ascending=True)['Genre'].tolist(),
    'NA Sales': genre_region.sort_values('NA_Sales', ascending=True)['Genre'].tolist(),
    'EU Sales': genre_region.sort_values('EU_Sales', ascending=True)['Genre'].tolist(),
    'JP Sales': genre_region.sort_values('JP_Sales', ascending=True)['Genre'].tolist(),}

color_map = {
    'North America': '#4C78A8',
    'Europe': '#F58518',
    'Japan': '#E45756',
    'Other': '#72B7B2'}

highlight = alt.selection_point(fields=['Genre'], on='click', empty='all')

sort_dropdown = alt.binding_select(
    options=list(sort_options.keys()),
    name='Sort by: ')

sort_param = alt.param(bind=sort_dropdown, value='Total Sales', name='sort_by')

bars = (
    alt.Chart(genre_long)
    .mark_bar()
    .encode(
        x=alt.X('Sales:Q', title='Total Sales (millions)'),
        y=alt.Y('Genre:N', title='Genre', sort=sort_options['Total Sales']),
        color=alt.Color('Region:N',
            scale=alt.Scale(domain=list(color_map.keys()), range=list(color_map.values())),
            legend=alt.Legend(title='Region')
        ),
        opacity=alt.condition(highlight, alt.value(1.0), alt.value(0.3)),
        tooltip=[
            alt.Tooltip('Genre:N', title='Genre'),
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('Sales:Q', title='Sales (M)', format='.1f'),
            alt.Tooltip('Pct:Q', title='% of Genre Total', format='.1f'),
            alt.Tooltip('Total:Q', title='Genre Total (M)', format='.1f'),
        ]
    )
    .add_params(highlight)
    .properties(width=700, height=400, title='Video Game Sales by Genre and Region'))
note = (
    alt.Chart({'values': [{}]})
    .mark_text(align='right', color='gray', fontSize=11)
    .encode(
        x=alt.value(700),
        y=alt.value(-10),
        text=alt.value('Click to highlight · Double-click to reset')))

(bars + note)


alt.LayerChart(...)